# Day 28 · From nudges to a formula: the chain rule, first taste

**First-principles question:** Can we get the exact slope without nudging?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 40 months of (fake) hosting data. The same every time you run it.
rng = np.random.default_rng(42)
x = rng.uniform(0.05, 2.0, 40).round(2)                     # requests per month, in lakhs (1 lakh = 100,000)
y = (2.5 + 1.8 * x + rng.normal(0, 0.25, 40)).round(2)      # hosting cost per month, in ₹ thousands
def predict(x, w, b):
    return w * x + b

def mse(w, b):
    return float(np.mean((predict(x, w, b) - y) ** 2))

H = 1e-5
def slope_w(w, b): return (mse(w + H, b) - mse(w, b)) / H
def slope_b(w, b): return (mse(w, b + H) - mse(w, b)) / H

## 1 · The exact gradient, with the chain rule
error = w·x + b − y, loss = mean(error²).

d loss/d w = mean(2 · error · x) and d loss/d b = mean(2 · error · 1).

In [ ]:
def gradient(w, b, x=x, y=y):
    """The exact (d loss / d w, d loss / d b)."""
    # TODO: error = predict(x, w, b) - y; return the two means from the formula above
    ...

print(gradient(1.0, 3.0))

## 2 · Gradient check
The exact gradient must agree with your nudges from day 3. Engineers run this check every time they write a gradient by hand.

In [ ]:
for w, b in [(0.0, 0.0), (1.0, 3.0), (2.5, 1.0)]:
    exact = gradient(w, b)
    nudged = (slope_w(w, b), slope_b(w, b))
    print(f"(w={w}, b={b}) exact {exact[0]:+.4f}, {exact[1]:+.4f} · nudged {nudged[0]:+.4f}, {nudged[1]:+.4f}")

## 3 · Which is faster on 100,000 points?

In [ ]:
import time
big_x = np.random.default_rng(1).uniform(0, 2, 100_000)
big_y = 2.5 + 1.8 * big_x
def big_mse(w, b): return float(np.mean((w * big_x + b - big_y) ** 2))

t0 = time.perf_counter(); gradient(1.0, 1.0, big_x, big_y); t_exact = time.perf_counter() - t0
t0 = time.perf_counter()
(big_mse(1 + H, 1) - big_mse(1, 1)) / H, (big_mse(1, 1 + H) - big_mse(1, 1)) / H
t_nudge = time.perf_counter() - t0
print(f"exact {t_exact * 1000:.1f} ms · nudging {t_nudge * 1000:.1f} ms (and nudging needs one extra pass PER parameter)")

## 4 · Train with the exact gradient

In [ ]:
w, b = 0.0, 0.0
for _ in range(500):
    gw, gb = gradient(w, b)
    w, b = w - 0.1 * gw, b - 0.1 * gb
print(f"w = {w:.3f}, b = {b:.3f}, loss = {mse(w, b):.4f}")

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
for w_, b_ in [(0.0, 0.0), (1.0, 3.0), (2.5, 1.0)]:
    gw, gb = gradient(w_, b_)
    assert abs(gw - slope_w(w_, b_)) < 1e-3 and abs(gb - slope_b(w_, b_)) < 1e-3, f"Gradient check failed at {(w_, b_)}"
assert abs(w - 1.8) < 0.2 and abs(b - 2.5) < 0.2
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")